In [1]:
import pandas as pd
import joblib
import json
from pathlib import Path
from pydantic import BaseModel, Field
from typing import List

PROJECT_DIR = Path('.').resolve().parent
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")

model = joblib.load(ARTIFACTS_DIR / 'final_model_random_forest.pkl')
scaler = joblib.load(ARTIFACTS_DIR / 'scaler.pkl')

with open(ARTIFACTS_DIR / 'final_model_report.json', 'r') as f:
    model_config = json.load(f)

print("Модель загружена")
print(f"  - Тип: {type(model).__name__}")
print(f"  - Параметры: {model.get_params()}")

print("\nScaler загружен")
print(f"  - Тип: {type(scaler).__name__}")

print("\nКонфиг модели загружен")
print(f"  - Features: {model_config['features_used']}")
print(f"  - CV ROC-AUC: {model_config['cv_results']['mean_roc_auc']:.4f}")
print(f"  - Test ROC-AUC: {model_config['test_results']['roc_auc']:.4f}")

PROJECT_DIR: C:\Users\kuzne\ai_engineering\project
ARTIFACTS_DIR: C:\Users\kuzne\ai_engineering\project\artifacts
Модель загружена
  - Тип: RandomForestClassifier
  - Параметры: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 20, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}

Scaler загружен
  - Тип: StandardScaler

Конфиг модели загружен
  - Features: ['age', 'income', 'employment_years', 'loan_amount', 'credit_score']
  - CV ROC-AUC: 0.9859
  - Test ROC-AUC: 0.9935


In [2]:
class CreditApplication(BaseModel):
    age: int = Field(..., ge=18, le=100, description="Age of applicant")
    income: float = Field(..., gt=0, description="Annual income")
    employment_years: int = Field(..., ge=0, le=50, description="Years of employment")
    loan_amount: float = Field(..., gt=0, description="Requested loan amount")
    credit_score: int = Field(..., ge=300, le=850, description="Credit score")
    
    class Config:
        json_schema_extra = {
            "example": {
                "age": 35,
                "income": 75000,
                "employment_years": 10,
                "loan_amount": 25000,
                "credit_score": 720
            }
        }

class PredictionResponse(BaseModel):
    prediction: int
    prediction_label: str
    probability_high_risk: float
    probability_low_risk: float
    risk_category: str

def create_prediction_pipeline(model, scaler, feature_cols: List[str]):
    def predict(application: CreditApplication) -> PredictionResponse:
        input_data = {
            'age': application.age,
            'income': application.income,
            'employment_years': application.employment_years,
            'loan_amount': application.loan_amount,
            'credit_score': application.credit_score
        }
        
        df = pd.DataFrame([input_data])
        df = df[feature_cols]
        
        X_scaled = scaler.transform(df)
        
        prediction = model.predict(X_scaled)[0]
        probabilities = model.predict_proba(X_scaled)[0]
        
        prob_high_risk = float(probabilities[1])
        prob_low_risk = float(probabilities[0])
        
        if prob_high_risk >= 0.7:
            risk_category = "HIGH"
        elif prob_high_risk >= 0.4:
            risk_category = "MEDIUM"
        else:
            risk_category = "LOW"
        
        return PredictionResponse(
            prediction=int(prediction),
            prediction_label="High Risk" if prediction == 1 else "Low Risk",
            probability_high_risk=round(prob_high_risk, 4),
            probability_low_risk=round(prob_low_risk, 4),
            risk_category=risk_category
        )
    
    return predict

pipeline = create_prediction_pipeline(model, scaler, model_config['features_used'])
print("Pipeline created successfully")

Pipeline created successfully


In [3]:
test_applications = [
    CreditApplication(age=35, income=75000, employment_years=10, loan_amount=25000, credit_score=720),
    CreditApplication(age=25, income=35000, employment_years=2, loan_amount=45000, credit_score=580),
    CreditApplication(age=50, income=120000, employment_years=20, loan_amount=15000, credit_score=800),
    CreditApplication(age=28, income=45000, employment_years=5, loan_amount=40000, credit_score=650),
    CreditApplication(age=60, income=95000, employment_years=30, loan_amount=10000, credit_score=750),
]

print("Testing prediction pipeline on sample applications\n")
print("=" * 100)
print(f"{'Age':<6} {'Income':<12} {'Loan':<12} {'Credit':<8} {'Prediction':<12} {'Prob(High)':<12} {'Risk':<8}")
print("=" * 100)

results = []
for app in test_applications:
    result = pipeline(app)
    results.append({
        'application': app,
        'prediction': result
    })
    
    print(f"{app.age:<6} {app.income:<12,} {app.loan_amount:<12,} {app.credit_score:<8} "
          f"{result.prediction_label:<12} {result.probability_high_risk:<12.4f} {result.risk_category:<8}")

print("=" * 100)

test_df = pd.DataFrame([{
    'age': r['application'].age,
    'income': r['application'].income,
    'employment_years': r['application'].employment_years,
    'loan_amount': r['application'].loan_amount,
    'credit_score': r['application'].credit_score,
    'prediction': r['prediction'].prediction_label,
    'probability_high_risk': r['prediction'].probability_high_risk,
    'risk_category': r['prediction'].risk_category
} for r in results])

print("\nDetailed results:")
display(test_df)

Testing prediction pipeline on sample applications

Age    Income       Loan         Credit   Prediction   Prob(High)   Risk    
35     75,000.0     25,000.0     720      Low Risk     0.0601       LOW     
25     35,000.0     45,000.0     580      High Risk    0.8937       HIGH    
50     120,000.0    15,000.0     800      Low Risk     0.0000       LOW     
28     45,000.0     40,000.0     650      High Risk    0.8832       HIGH    
60     95,000.0     10,000.0     750      Low Risk     0.0000       LOW     

Detailed results:


,age,income,employment_years,loan_amount,credit_score,prediction,probability_high_risk,risk_category
0,35,75000.0,10,25000.0,720,Low Risk,0.0601,LOW
1,25,35000.0,2,45000.0,580,High Risk,0.8937,HIGH
2,50,120000.0,20,15000.0,800,Low Risk,0.0000,LOW
3,28,45000.0,5,40000.0,650,High Risk,0.8832,HIGH
4,60,95000.0,30,10000.0,750,Low Risk,0.0000,LOW


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report, confusion_matrix
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path('.').resolve().parent
DATA_PATH = PROJECT_DIR / "data" / "credit_risk_data.csv"

df = pd.read_csv(DATA_PATH)
feature_cols = ['age', 'income', 'employment_years', 'loan_amount', 'credit_score']
X = df[feature_cols]
y = df['target']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print("Размеры выборок:")
print(f"  Train: {len(X_train)} ({len(X_train)/len(df)*100:.0f}%)")
print(f"  Val:   {len(X_val)} ({len(X_val)/len(df)*100:.0f}%)")
print(f"  Test:  {len(X_test)} ({len(X_test)/len(df)*100:.0f}%)")

best_params = {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
final_model = RandomForestClassifier(**best_params, random_state=42, class_weight='balanced', n_jobs=-1)

X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

final_model.fit(X_train_val, y_train_val)

# Шаг 4: Оценка на строго изолированном TEST set
y_pred_test = final_model.predict(X_test)
y_prob_test = final_model.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("ФИНАЛЬНАЯ ОЦЕНКА НА ИЗОЛИРОВАННОМ TEST SET")
print("="*60)
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_test):.4f}")
print(f"F1-score (High Risk): {f1_score(y_test, y_pred_test):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_test, target_names=['Low Risk', 'High Risk']))

conf_matrix = confusion_matrix(y_test, y_pred_test)
print("Confusion Matrix:")
print("                Pred Low   Pred High")
print(f"Actual Low      {conf_matrix[0,0]:<10} {conf_matrix[0,1]}")
print(f"Actual High     {conf_matrix[1,0]:<10} {conf_matrix[1,1]}")

print(f"\nFalse Negatives (пропущенный дефолт): {conf_matrix[1,0]}")
print(f"False Positives (ложная тревога): {conf_matrix[0,1]}")

Размеры выборок:
  Train: 600 (60%)
  Val:   200 (20%)
  Test:  200 (20%)

ФИНАЛЬНАЯ ОЦЕНКА НА ИЗОЛИРОВАННОМ TEST SET
ROC-AUC: 0.9905
F1-score (High Risk): 0.9076
Accuracy: 0.9450

Classification Report:
              precision    recall  f1-score   support

    Low Risk       0.96      0.96      0.96       140
   High Risk       0.92      0.90      0.91        60

    accuracy                           0.94       200
   macro avg       0.94      0.93      0.93       200
weighted avg       0.94      0.94      0.94       200

Confusion Matrix:
                Pred Low   Pred High
Actual Low      135        5
Actual High     6          54

False Negatives (пропущенный дефолт): 6
False Positives (ложная тревога): 5


## Результаты финального тестирования

### Корректное разделение данных
- Train: 600 samples (60%) - обучение
- Val: 200 samples (20%) - валидация/настройка гиперпараметров
- Test: 200 samples (20%) - строго изолированная финальная оценка

### Производительность на изолированном Test set
- ROC-AUC: 0.9905
- F1-score (High Risk): 0.9076
- Accuracy: 0.9450

### Confusion Matrix
- True Negatives: 135 (корректно Low Risk)
- True Positives: 54 (корректно High Risk)
- False Positives: 5 (ложная тревога)
- False Negatives: 6 (пропущенный дефолт)

### Анализ ошибок
**False Negatives (6 случаев)**: модель пропустила 6 реальных дефолтов из 60
- Это 10% от всех High Risk случаев
- Критично для бизнеса: не выявленные рискованные заемщики

**False Positives (5 случаев)**: модель ошибочно пометила 5 хороших заемщиков
- Это 3.7% от всех Low Risk случаев
- Менее критично: потеря потенциальных клиентов

### Интерпретация
Модель показывает отличные результаты:
- ROC-AUC 0.9905 подтверждает высокую дискриминационную способность
- F1 0.9076 для minority class - хороший баланс precision/recall
- Модель готова к продакшену с пониманием ограничений


In [8]:
import json
from datetime import datetime

pipeline_test_results = {
    'timestamp': datetime.now().isoformat(),
    'model_name': model_config['model_name'],
    'test_performance': {
        'roc_auc': float(roc_auc_score(y_test_full, y_prob_test)),
        'f1': float(f1_score(y_test_full, y_pred_test)),
        'accuracy': float(accuracy_score(y_test_full, y_pred_test)),
        'confusion_matrix': {
            'true_negatives': int(conf_matrix[0,0]),
            'false_positives': int(conf_matrix[0,1]),
            'false_negatives': int(conf_matrix[1,0]),
            'true_positives': int(conf_matrix[1,1])
        }
    },
    'sample_predictions': [{
        'input': {
            'age': r['application'].age,
            'income': r['application'].income,
            'employment_years': r['application'].employment_years,
            'loan_amount': r['application'].loan_amount,
            'credit_score': r['application'].credit_score
        },
        'output': {
            'prediction': r['prediction'].prediction_label,
            'probability_high_risk': r['prediction'].probability_high_risk,
            'risk_category': r['prediction'].risk_category
        }
    } for r in results],
    'pipeline_ready': True,
    'validation_schema': 'CreditApplication (Pydantic)',
    'output_schema': 'PredictionResponse (Pydantic)'
}

pipeline_report_path = ARTIFACTS_DIR / 'pipeline_test_results.json'
with open(pipeline_report_path, 'w') as f:
    json.dump(pipeline_test_results, f, indent=2)

print(f"Pipeline test results saved: {pipeline_report_path}")

print("\nAll artifacts in artifacts/:")
artifact_files = [f.name for f in ARTIFACTS_DIR.iterdir() if f.is_file()]
for f in sorted(artifact_files):
    print(f"  - {f}")

Pipeline test results saved: C:\Users\kuzne\ai_engineering\project\artifacts\pipeline_test_results.json

All artifacts in artifacts/:
  - README.md
  - correlation_matrix.png
  - cv_scores.csv
  - data_quality_report.json
  - eda_boxplots_outliers.png
  - eda_correlation_matrix.png
  - eda_numeric_distributions.png
  - eda_pairplot_key_features.png
  - eda_summary_report.json
  - eda_target_correlation.png
  - eda_target_distribution.png
  - final_model_random_forest.pkl
  - final_model_report.json
  - model_comparison_results.json
  - model_feature_importance.png
  - pipeline_test_results.json
  - scaler.pkl
  - target_distribution.png
